# Main run, t-test, epsilon trade-off, physical air-gap test

PHC-351 · Hiding a Voice · Bhavishay (24324008), IIT Roorkee

Code files 09_main_run.py, 10_level_norm_and_ttest.py, 11_eps_curve.py, 12_airgap.py from `code/`, as Colab cells in run order.

**How to run:** Runtime → Change runtime type → **T4 GPU** → Save, then **Runtime → Run all**. Allow Google Drive access when asked.

Needs the project folder on your Google Drive. 09 protects 20 speakers through 4 held-out rooms, 11 the trade-off; both take a while on a T4. The optimisation is not seeded, so the numbers come out close to, not identical with, the paper.

**Changes from the original cells, so that the notebook runs on its own:** a setup cell at the top; cells marked `ADDED for the re-run` (names the original session defined in cells that were not kept, and two result printouts); and file paths, marked `# re-run:`, so that everything is written into a run folder in this Colab session and the files on Drive are never overwritten. Everything else is the code as run; `code/` keeps the original paths.

## Setup (added)

In [ ]:
# ===== SETUP (added so this notebook runs top to bottom from GitHub; not part of the original runs) =====
import os, glob, shutil, torch, torchaudio, numpy as np, soundfile as sf
RUN = '/content/PHC351_rerun'          # this session's working folder; Colab deletes it when the session ends
os.makedirs(RUN, exist_ok=True)

# Audio files: newer torchaudio needs the extra package torchcodec for load/save.
# If it is not installed, load/save go through soundfile instead (same samples, same shapes).
try:
    torchaudio.save(f'{RUN}/_probe.wav', torch.zeros(1, 160), 16000); torchaudio.load(f'{RUN}/_probe.wav')
except Exception:
    def _load(path, *a, **k):
        x, sr = sf.read(path, dtype='float32', always_2d=True)
        return torch.from_numpy(np.ascontiguousarray(x.T)), sr
    def _save(path, x, sr, *a, **k):
        sf.write(path, np.ascontiguousarray(x.detach().cpu().numpy().T), sr, subtype='FLOAT')
    torchaudio.load, torchaudio.save = _load, _save
    print('audio load/save: soundfile (torchcodec not installed)')

NEED = ['code/modelA_speaker_encoder.pt', 'code/modelB_independent_judge.pt', 'code/modelC_ensemble_member.pt', 'testing_data/recordings/measured_rirs.pt', 'testing_data/results/thresholds_measured.pt', 'demo_best_config/A_original.wav', 'training_data/heldout_subset/*.flac', 'testing_data/recordings/clean2_*.wav', 'testing_data/recordings/prot2_*.wav']
# Inputs: read from the project folder on Google Drive and copied into RUN.
# The code below writes only into RUN, so the original files on Drive are never overwritten.
from google.colab import drive
drive.mount('/content/drive')
ORIG = '/content/drive/MyDrive/PHC351_24324008_VoiceProtection'
if not os.path.isdir(ORIG):
    raise SystemExit(f'Project folder not found on this Google Drive: {ORIG}\n'
                     'This notebook needs the recordings and models in that folder (the recordings are not in the repository).')
n = 0
for sub, pat in [('code', '*.pt'), ('testing_data/recordings', '*.wav'), ('testing_data/recordings', '*.pt'),
                 ('testing_data/results', 'thresholds_measured.pt'), ('training_data/heldout_subset', '*.flac'),
                 ('demo_best_config', '*.wav')]:
    os.makedirs(f'{RUN}/{sub}', exist_ok=True)
    for f in glob.glob(f'{ORIG}/{sub}/{pat}'):
        dst = f'{RUN}/{sub}/{os.path.basename(f)}'
        if not os.path.exists(dst): shutil.copy2(f, dst); n += 1
os.makedirs(f'{RUN}/figures', exist_ok=True)
print(f'copied {n} input files from the project folder into {RUN} (the project folder is only read)')
missing = [p for p in NEED if not glob.glob(f'{RUN}/{p}')]
print('all inputs found' if not missing else f'MISSING inputs (the cells that use them will fail): {missing}')

## 09_main_run.py

Restore from Drive, protect_meas, and the main run: 20 speakers x 4 held-out rooms, eps 0.004 (98% -> 28%).

In [ ]:
# ============ RESTORE + MULTI-SPEAKER RUN ============
import os, glob, math, random, time, json, torch, torchaudio, torch.nn as nn
import numpy as np
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev); assert dev == 'cuda', 'STOP — switch to T4 GPU first'
!pip install -q pesq
from pesq import pesq as pesq_fn

R = RUN   # re-run: run folder (original: '/content/drive/MyDrive/PHC351_24324008_VoiceProtection')
REC, RES, FIG = f'{R}/testing_data/recordings', f'{R}/testing_data/results', f'{R}/figures'
SR, N_FFT, HOP, WIN, N_MELS = 16000, 512, 160, 400, 80
CHUNK = 3*SR
_w  = torch.hann_window(WIN)
_fb = torchaudio.functional.melscale_fbanks(N_FFT//2+1, 0., SR/2, N_MELS, SR,
                                            norm=None, mel_scale="htk")
def logmel(x):
    X = torch.stft(x, N_FFT, HOP, WIN, _w.to(x.device), center=True, return_complex=True)
    return torch.log(_fb.T.to(x.device) @ (X.abs()**2) + 1e-6)

class SpeakerNet(nn.Module):
    def __init__(s, nc, d=192):
        super().__init__()
        b = lambda ci,co,st: nn.Sequential(nn.Conv2d(ci,co,3,st,1), nn.BatchNorm2d(co), nn.ReLU(True))
        s.conv = nn.Sequential(b(1,32,(2,1)), b(32,64,(2,1)), b(64,128,(2,1)), b(128,128,(2,1)))
        s.proj = nn.Linear(128*5*2, d); s.bn = nn.BatchNorm1d(d); s.head = nn.Linear(d, nc)
    def forward(s, x):
        h = s.conv(x.unsqueeze(1)); B,C,F,T = h.shape; h = h.reshape(B, C*F, T)
        e = s.bn(s.proj(torch.cat([h.mean(2), h.var(2,unbiased=False).clamp_min(1e-8).sqrt()],1)))
        return e, s.head(e)

class SpeakerNetB(nn.Module):
    def __init__(s, nc, d=256):
        super().__init__()
        b = lambda ci,co,k,st: nn.Sequential(nn.Conv2d(ci,co,k,st,k//2), nn.BatchNorm2d(co), nn.ReLU(True))
        s.conv = nn.Sequential(b(1,48,5,(2,1)), b(48,96,3,(2,1)), b(96,160,3,(2,2)),
                               b(160,160,3,(2,1)), b(160,160,3,(1,1)))
        s.proj = nn.Sequential(nn.Linear(160*5*2,512), nn.ReLU(True), nn.Linear(512,d))
        s.bn = nn.BatchNorm1d(d); s.head = nn.Linear(d, nc)
    def forward(s, x):
        h = s.conv(x.unsqueeze(1)); B,C,F,T = h.shape; h = h.reshape(B, C*F, T)
        e = s.bn(s.proj(torch.cat([h.mean(2), h.var(2,unbiased=False).clamp_min(1e-8).sqrt()],1)))
        return e, s.head(e)

def load_net(cls, pats):
    for pat in pats:
        hits = glob.glob(f'{R}/**/{pat}', recursive=True)
        if hits:
            f_ = sorted(hits)[-1]; sd = torch.load(f_, map_location=dev)
            sd = sd['m'] if isinstance(sd, dict) and 'm' in sd else sd
            net = cls(sd['head.weight'].shape[0]).to(dev); net.load_state_dict(sd)
            print(f'  {cls.__name__:12s} <- {os.path.basename(f_)}'); return net.eval()
    print(f'  !! {cls.__name__} NOT FOUND'); return None
model  = load_net(SpeakerNet,  ['modelA*.pt'])
modelB = load_net(SpeakerNetB, ['modelB_independent*.pt','modelB*.pt'])

def load16k(p):
    x, sr = torchaudio.load(p); x = x.mean(0)
    return torchaudio.functional.resample(x, sr, SR) if sr != SR else x
def fit(x, n=CHUNK): return torch.nn.functional.pad(x,(0,max(0,n-x.shape[0])))[:n]
def apply_rir(x, h):
    L = x.shape[0]+h.shape[0]-1; nf = 1 << (L-1).bit_length()
    y = torch.fft.irfft(torch.fft.rfft(x,nf)*torch.fft.rfft(h.to(x.device),nf), nf)[:x.shape[0]]
    return y / y.abs().max().clamp_min(1e-9) * x.abs().max()
def emb(net, x):
    with torch.no_grad(): e,_ = net(logmel(x.to(dev)).unsqueeze(0))
    return torch.nn.functional.normalize(e, dim=1)[0]
cos    = lambda a,b: float((a*b).sum())
snr_db = lambda x,d: 20*torch.log10(x.std()/d.std().clamp_min(1e-9)).item()
nrm    = lambda v: torch.nn.functional.normalize(v, dim=0)
def pq(ref, deg):
    try: return pesq_fn(SR, ref.numpy(), deg.numpy(), 'wb')
    except Exception: return float('nan')

def protect_meas(wav, eps, models, K=300, eot=4, rir_pool=None):
    wav = wav.to(dev); alpha = eps/10
    pool = [h.to(dev) for h in rir_pool]
    e0 = [emb(M, wav) for M in models]
    d = torch.zeros_like(wav, requires_grad=True)
    for _ in range(K):
        x = (wav + d).clamp(-1,1); tot = 0
        for _b in range(eot):
            xb = apply_rir(x, pool[random.randrange(len(pool))])
            for M, a0 in zip(models, e0):
                ee = torch.nn.functional.normalize(M(logmel(xb).unsqueeze(0))[0], dim=1)[0]
                tot = tot + (ee*a0).sum()
        g, = torch.autograd.grad(tot/(eot*len(models)), d)
        d = (d - alpha*g.sign()).clamp(-eps, eps).detach().requires_grad_(True)
    return (wav + d).clamp(-1,1).detach().cpu()

md = torch.load(f'{REC}/measured_rirs.pt'); RIR_MEAS, ready = md['rirs'], md['rooms']
th = torch.load(f'{RES}/thresholds_measured.pt'); TAU_A, TAU_B = th['TAU_A'], th['TAU_B']
src = fit(load16k(f'{R}/demo_best_config/A_original.wav'))
held = {}
for p in sorted(glob.glob(f'{R}/training_data/heldout_subset/*.flac')):
    held.setdefault(os.path.basename(p).split('_')[0], []).append(p)
held = {k:v for k,v in held.items() if len(v) >= 3}
print(f'rooms {ready}   tau_B {TAU_B:.3f}   {len(held)} cached held-out speakers')

# ---- save the operating-point clip for re-recording ----
EPS_OP = 0.004
w_op = protect_meas(src, EPS_OP, [model], rir_pool=list(RIR_MEAS.values()))
torchaudio.save(f'{R}/demo_best_config/C_protected_eps004.wav', w_op.unsqueeze(0), SR)
print(f'saved C_protected_eps004.wav   SNR {snr_db(src, w_op-src):.1f} dB  PESQ {pq(src, w_op):.2f}')

# ---- multi-speaker ----
NSPK = 20
spk_run = [s for s in held][:NSPK]
res, t0 = [], time.time()
for si, s in enumerate(spk_run):
    ps = random.Random(si).sample(held[s], 2)
    s_src, s_enr = fit(load16k(ps[0])), fit(load16k(ps[1]))
    for hr in ready:
        pool  = [RIR_MEAS[r] for r in ready if r != hr]
        train = [r for r in ready if r != hr]; ht = RIR_MEAS[hr]
        EB = nrm(torch.stack([emb(modelB, apply_rir(s_enr, RIR_MEAS[r])) for r in train]
                             + [emb(modelB, s_enr)]).mean(0))
        EA = nrm(torch.stack([emb(model,  apply_rir(s_enr, RIR_MEAS[r])) for r in train]
                             + [emb(model,  s_enr)]).mean(0))
        w  = protect_meas(s_src, EPS_OP, [model], rir_pool=pool)
        yc, yp = apply_rir(s_src, ht), apply_rir(w, ht)
        res.append(dict(spk=s, room=hr,
                        cleanB=cos(EB, emb(modelB, yc)), protB=cos(EB, emb(modelB, yp)),
                        cleanA=cos(EA, emb(model,  yc)), protA=cos(EA, emb(model,  yp)),
                        snr=snr_db(s_src, w-s_src), pesq=pq(s_src, w)))
    json.dump(res, open(f'{RES}/multispeaker.json','w'))
    print(f'  {si+1:2d}/{NSPK}  {s:>8s}  {(time.time()-t0)/60:5.1f} min')

A = {k: np.array([r[k] for r in res]) for k in ('cleanB','protB','cleanA','protA','snr','pesq')}
ci = lambda v: 1.96*v.std(ddof=1)/np.sqrt(len(v))
print(f'\nn = {len(res)} trials ({NSPK} speakers x {len(ready)} rooms), eps={EPS_OP}')
for k in ('cleanB','protB','cleanA','protA'):
    print(f'  {k:8s} {A[k].mean():.3f} +- {ci(A[k]):.3f}')
print(f'  SNR {A["snr"].mean():.1f} dB   PESQ {np.nanmean(A["pesq"]):.2f}')
print(f'  attacker succeeds (cosB > {TAU_B:.3f}):  clean {100*(A["cleanB"]>TAU_B).mean():.0f}%'
      f'   protected {100*(A["protB"]>TAU_B).mean():.0f}%')

## 10_level_norm_and_ttest.py

Paired t-test on the 80 trials (t = 18.92), and the level-normalised clip used in the rooms and the listening test (fix for C7).

In [ ]:
from scipy import stats
d = A['cleanB'] - A['protB']
t, p = stats.ttest_rel(A['cleanB'], A['protB'])
print(f'paired t = {t:.2f}, p = {p:.2e}, mean drop {d.mean():.3f} +- {ci(d):.3f}')
print(f"Cohen's d = {d.mean()/d.std(ddof=1):.2f}")

In [ ]:
# ===== next cell =====
src_n = src / src.std() * 0.05                      # standard level
w_op  = protect_meas(src_n, 0.004, [model], rir_pool=list(RIR_MEAS.values()))
torchaudio.save(f'{R}/demo_best_config/C_protected_eps004.wav', w_op.unsqueeze(0), SR)
torchaudio.save(f'{R}/demo_best_config/C_original_norm.wav', src_n.unsqueeze(0), SR)
print(f'SNR {snr_db(src_n, w_op-src_n):.1f} dB   PESQ {pq(src_n, w_op):.2f}')

## 11_eps_curve.py

protect_meas and the epsilon curve over the four held-out rooms (encoder A, and A + C).

In [ ]:
# ===== ADDED for the re-run: encoder C, used by 11; it was trained by 14_model_C_and_ensemble.py and saved in the project folder =====
class SpeakerNetC(nn.Module):
    """Deliberately unlike A and B: dilated convs, wider, 320-dim."""
    def __init__(s, nc, d=320):
        super().__init__()
        b = lambda ci,co,k,st,dl=1: nn.Sequential(
            nn.Conv2d(ci,co,k,st,(k//2)*dl,dilation=dl), nn.BatchNorm2d(co), nn.ReLU(True))
        s.conv = nn.Sequential(b(1,64,3,(2,1)), b(64,96,3,(2,1),2), b(96,128,3,(2,1),2),
                               b(128,160,3,(2,2)), b(160,192,3,(1,1),4))
        s.proj = nn.Sequential(nn.Linear(192*5*2,640), nn.ReLU(True), nn.Linear(640,d))
        s.bn = nn.BatchNorm1d(d); s.head = nn.Linear(d, nc)
    def forward(s, x):
        h = s.conv(x.unsqueeze(1)); B,C,F,T = h.shape; h = h.reshape(B, C*F, T)
        e = s.bn(s.proj(torch.cat([h.mean(2),
              h.var(2, unbiased=False).clamp_min(1e-8).sqrt()], 1)))
        return e, s.head(e)

_sd = torch.load(f'{R}/code/modelC_ensemble_member.pt', map_location=dev)
modelC = SpeakerNetC(_sd['head.weight'].shape[0]).to(dev); modelC.load_state_dict(_sd); modelC.eval()

In [ ]:
from pesq import pesq as pesq_fn

def protect_meas(wav, eps, models, K=300, eot=4, rir_pool=None):
    """plain measured-channel EOT: untargeted, no momentum, no diversity, no mask"""
    wav = wav.to(dev); alpha = eps/10
    pool = [h.to(dev) for h in rir_pool]
    e0 = [emb(M, wav) for M in models]
    d = torch.zeros_like(wav, requires_grad=True)
    for _ in range(K):
        x = (wav + d).clamp(-1,1); tot = 0
        for _b in range(eot):
            xb = apply_rir(x, pool[random.randrange(len(pool))])
            for M, a0 in zip(models, e0):
                ee = torch.nn.functional.normalize(M(logmel(xb).unsqueeze(0))[0], dim=1)[0]
                tot = tot + (ee*a0).sum()
        g, = torch.autograd.grad(tot/(eot*len(models)), d)
        d = (d - alpha*g.sign()).clamp(-eps, eps).detach().requires_grad_(True)
    return (wav + d).clamp(-1,1).detach().cpu()

def pq(ref, deg):
    try: return pesq_fn(SR, ref.numpy(), deg.numpy(), 'wb')
    except Exception: return float('nan')

nrm = lambda v: torch.nn.functional.normalize(v, dim=0)
torch.manual_seed(0); random.seed(0)
EPSL = [0.002, 0.005, 0.010, 0.020]
rows = []
for nm, ms in [('A', [model]), ('A+C', [model, modelC])]:
    for eps in EPSL:
        cB, cA, sn, pe = [], [], [], []
        for hr in ready:
            pool  = [RIR_MEAS[r] for r in ready if r != hr]
            train = [r for r in ready if r != hr]
            E = {k: nrm(torch.stack([emb(M, apply_rir(src, RIR_MEAS[r])) for r in train]
                                    + [emb(M, src)]).mean(0))
                 for k, M in [('A',model),('B',modelB)]}
            w = protect_meas(src, eps, ms, rir_pool=pool)
            y = apply_rir(w, RIR_MEAS[hr])
            cB.append(cos(E['B'], emb(modelB, y))); cA.append(cos(E['A'], emb(model, y)))
            sn.append(snr_db(src, w - src)); pe.append(pq(src, w))
        rows.append((nm, eps, np.mean(sn), np.nanmean(pe), np.mean(cB), np.mean(cA)))

In [ ]:
# ===== ADDED for the re-run: print the table computed by 11 =====
print(f'{"models":7s}{"eps":>7s}{"SNR dB":>8s}{"PESQ":>7s}{"cosB":>8s}{"cosA":>8s}')
for nm, eps, sn, pe, cB, cA in rows:
    print(f'{nm:7s}{eps:7.3f}{sn:8.1f}{pe:7.2f}{cB:8.3f}{cA:8.3f}')

## 12_airgap.py

The physical air-gap result: the clips played aloud in 4 rooms and re-recorded on the phone (100% -> 75%).

In [ ]:
# ===== ADDED for the re-run: align_to from 07_rooms.py, used by 12 =====
def align_to(rec, ref):
    n = 1 << (rec.shape[0] + ref.shape[0] - 1).bit_length()
    c = torch.fft.irfft(torch.fft.rfft(rec, n) * torch.fft.rfft(ref.flip(0), n), n)
    lag = max(0, int(c[:rec.shape[0]].argmax()) - (ref.shape[0] - 1))
    y = rec[lag : lag + ref.shape[0]]
    y = torch.nn.functional.pad(y, (0, max(0, ref.shape[0] - y.shape[0])))
    return y * (ref.std() / y.std().clamp_min(1e-9))

In [ ]:
md = torch.load(f'{REC}/measured_rirs.pt'); RIR_MEAS, ready = md['rirs'], md['rooms']
th = torch.load(f'{RES}/thresholds_measured.pt'); TAU_A, TAU_B = th['TAU_A'], th['TAU_B']
src_n = fit(load16k(f'{R}/demo_best_config/C_original_norm.wav'))
w_op  = fit(load16k(f'{R}/demo_best_config/C_protected_eps004.wav'))

have = {os.path.basename(q)[:-4] for q in glob.glob(f'{REC}/*.wav')}
air = [r for r in ready if f'clean2_{r}' in have and f'prot2_{r}' in have]

print(f'\n{"room":15s}{"clean":>9s}{"prot":>9s}{"drop":>8s}   judge / enrolment')
rows = []
for r in air:
    tr = [q for q in air if q != r]
    rc = align_to(load16k(f'{REC}/clean2_{r}.wav'), src_n)
    rp = align_to(load16k(f'{REC}/prot2_{r}.wav'),  w_op)
    for jn, J in [('B', modelB), ('A', model)]:
        # (i) enrolment from measured RIRs of the other rooms
        Ed = nrm(torch.stack([emb(J, apply_rir(src_n, RIR_MEAS[q])) for q in tr]
                             + [emb(J, src_n)]).mean(0))
        # (ii) enrolment from the actual clean recordings in the other rooms
        Ep = nrm(torch.stack([emb(J, align_to(load16k(f'{REC}/clean2_{q}.wav'), src_n))
                              for q in tr]).mean(0))
        for en, E in [('simulated', Ed), ('recorded', Ep)]:
            c1, c2 = cos(E, emb(J, rc)), cos(E, emb(J, rp))
            rows.append(dict(room=r, judge=jn, enrol=en, clean=c1, prot=c2))
            print(f'{r:15s}{c1:9.3f}{c2:9.3f}{c1-c2:8.3f}   judge {jn} / {en}')
json.dump(rows, open(f'{RES}/airgap_final.json','w'))
b = [x for x in rows if x['judge']=='B' and x['enrol']=='recorded']
print(f'\nJUDGE B, recorded enrolment  (tau_B = {TAU_B:.3f})')
print(f'  clean     {np.mean([x["clean"] for x in b]):.3f}'
      f'   attacker succeeds {100*np.mean([x["clean"]>TAU_B for x in b]):.0f}%')
print(f'  protected {np.mean([x["prot"]  for x in b]):.3f}'
      f'   attacker succeeds {100*np.mean([x["prot"] >TAU_B for x in b]):.0f}%')